# Baseball Win Probability — Logistic Regression Model

Binary classification: predict home team win from pre-game starting lineup stats.  
Train: 2023–2024 seasons | Test: 2025 season

In [ ]:
from pathlib import Path

# ── Data ────────────────────────────────────────────────────────────────────
DATA_DIR      = Path("data")
TRAIN_PATH    = DATA_DIR / "train_features.parquet"
TEST_PATH     = DATA_DIR / "test_features.parquet"
METADATA_COLS = ["date", "home_team", "away_team", "home_win"]

# ── Feature Engineering ──────────────────────────────────────────────────────
# True  → compute home-minus-away differentials (59 features, recommended)
# False → use all raw home_* and away_* features (118 features)
USE_DIFFERENTIAL_FEATURES = True

# ── Model Hyperparameters ────────────────────────────────────────────────────
MODEL_CONFIG = {
    "solver":       "lbfgs",   # 'lbfgs' | 'liblinear' | 'saga' | 'newton-cg'
    "penalty":      "l2",      # 'l1' | 'l2' | 'elasticnet' | None
    "C":            1.0,       # inverse regularization strength; smaller = stronger
    "max_iter":     1000,
    "class_weight": None,      # None | 'balanced'
    "random_state": 42,
    "l1_ratio":     None,      # only used when penalty='elasticnet' and solver='saga'
}

# ── Evaluation ───────────────────────────────────────────────────────────────
CALIBRATION_BINS = 10   # number of bins for calibration plot
TOP_N_FEATURES   = 20   # number of top coefficients to show in importance plot

## 1. Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    RocCurveDisplay,
    brier_score_loss,
)
from sklearn.calibration import calibration_curve

sns.set_theme(style="whitegrid")

## 2. Load Data

In [ ]:
train_df = pd.read_parquet(TRAIN_PATH)
test_df  = pd.read_parquet(TEST_PATH)

print(f"Train: {train_df.shape}  ({train_df['date'].dt.year.min()}-{train_df['date'].dt.year.max()})")
print(f"Test:  {test_df.shape}   ({test_df['date'].dt.year.min()}-{test_df['date'].dt.year.max()})")
print(f"\nTarget balance (train): {train_df['home_win'].mean():.3f} home win rate")
print(f"Target balance (test):  {test_df['home_win'].mean():.3f} home win rate")

feat_cols = [c for c in train_df.columns if c not in METADATA_COLS]
print(f"\nFeature columns: {len(feat_cols)}")
train_df[feat_cols].describe().T.head(10)

## 3. Feature Engineering

Two modes (controlled by `USE_DIFFERENTIAL_FEATURES`):

- **Differential** (default): for each matched player slot, compute `home_stat - away_stat`.  
  Produces 59 features. Positive values mean home team advantage.  
  For ERA and WHIP (lower = better for pitchers), the sign is flipped: `away_sp_ERA - home_sp_ERA`.

- **Raw**: keep all 118 raw player-stat columns unchanged.

In [ ]:
BATTING_STATS  = ["BA", "OBP", "SLG", "OPS", "K%", "BB%"]
PITCHING_STATS = ["ERA", "WHIP", "SO9", "SO/W", "IP"]
PITCHING_LOWER_IS_BETTER = {"ERA", "WHIP"}   # flip sign so positive = home advantage


def make_differential_features(df):
    """Return a DataFrame of home-minus-away differential features.

    Batting:  diff_bat{N}_{stat} = home_bat{N}_{stat} - away_bat{N}_{stat}
    Pitching: diff_sp_{stat}     = home_sp_{stat} - away_sp_{stat}   (SO9, SO/W, IP)
              diff_sp_{stat}     = away_sp_{stat} - home_sp_{stat}   (ERA, WHIP)
    All columns are oriented so positive = home team advantage.
    """
    diffs = {}
    for n in range(1, 10):
        for stat in BATTING_STATS:
            diffs[f"diff_bat{n}_{stat}"] = df[f"home_bat{n}_{stat}"] - df[f"away_bat{n}_{stat}"]
    for stat in PITCHING_STATS:
        if stat in PITCHING_LOWER_IS_BETTER:
            diffs[f"diff_sp_{stat}"] = df[f"away_sp_{stat}"] - df[f"home_sp_{stat}"]
        else:
            diffs[f"diff_sp_{stat}"] = df[f"home_sp_{stat}"] - df[f"away_sp_{stat}"]
    return pd.DataFrame(diffs, index=df.index)

In [ ]:
if USE_DIFFERENTIAL_FEATURES:
    X_train = make_differential_features(train_df)
    X_test  = make_differential_features(test_df)
    print(f"Using differential features: {X_train.shape[1]} columns")
else:
    X_train = train_df[feat_cols].copy()
    X_test  = test_df[feat_cols].copy()
    print(f"Using raw features: {X_train.shape[1]} columns")

y_train = train_df["home_win"].values
y_test  = test_df["home_win"].values

print(f"X_train: {X_train.shape} | X_test: {X_test.shape}")
assert X_train.isna().sum().sum() == 0, "NaNs found in training features"
assert X_test.isna().sum().sum()  == 0, "NaNs found in test features"

## 4. Preprocessing

Logistic regression is sensitive to feature scale. `StandardScaler` is fit on training data only
and applied to both train and test sets to avoid data leakage.

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)   # fit only on train

print("Feature scale after StandardScaler:")
print(f"  Train mean range: [{X_train_scaled.mean(axis=0).min():.4f}, {X_train_scaled.mean(axis=0).max():.4f}]")
print(f"  Train std  range: [{X_train_scaled.std(axis=0).min():.4f},  {X_train_scaled.std(axis=0).max():.4f}]")

## 5. Train Model

In [ ]:
lr_kwargs = {k: v for k, v in MODEL_CONFIG.items() if v is not None}
if MODEL_CONFIG["penalty"] != "elasticnet":
    lr_kwargs.pop("l1_ratio", None)

model = LogisticRegression(**lr_kwargs)
model.fit(X_train_scaled, y_train)

print(f"Solver:   {MODEL_CONFIG['solver']}")
print(f"Penalty:  {MODEL_CONFIG['penalty']}  (C={MODEL_CONFIG['C']})")
print(f"Converged in {model.n_iter_[0]} iterations (max_iter={MODEL_CONFIG['max_iter']})")
if model.n_iter_[0] == MODEL_CONFIG["max_iter"]:
    print("WARNING: max_iter reached -- consider increasing max_iter or adjusting solver")

## 6. Evaluation

In [ ]:
y_pred_train = model.predict(X_train_scaled)
y_pred_test  = model.predict(X_test_scaled)
y_prob_train = model.predict_proba(X_train_scaled)[:, 1]
y_prob_test  = model.predict_proba(X_test_scaled)[:, 1]

metrics = {
    "Accuracy":    (accuracy_score(y_train, y_pred_train),
                    accuracy_score(y_test,  y_pred_test)),
    "ROC-AUC":     (roc_auc_score(y_train, y_prob_train),
                    roc_auc_score(y_test,  y_prob_test)),
    "Brier Score": (brier_score_loss(y_train, y_prob_train),
                    brier_score_loss(y_test,  y_prob_test)),
}

metrics_df = pd.DataFrame(metrics, index=["Train", "Test"]).T
print(metrics_df.to_string())

In [ ]:
print("=== Classification Report (Test) ===")
print(classification_report(y_test, y_pred_test, target_names=["Away Win", "Home Win"]))

fig, ax = plt.subplots(figsize=(5, 4))
cm = confusion_matrix(y_test, y_pred_test)
ConfusionMatrixDisplay(cm, display_labels=["Away Win", "Home Win"]).plot(ax=ax, colorbar=False)
ax.set_title("Confusion Matrix -- Test Set")
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
RocCurveDisplay.from_predictions(y_test, y_prob_test, name="Logistic Regression", ax=ax)
ax.plot([0, 1], [0, 1], "k--", label="Random")
ax.set_title("ROC Curve -- Test Set")
ax.legend()
plt.tight_layout()
plt.show()

## 7. Feature Importance

Coefficients reflect the log-odds contribution of each standardized feature.  
For differential features, positive = home team advantage on that stat.

In [ ]:
feature_names = list(X_train.columns)
coefs = model.coef_[0]

coef_df = (
    pd.DataFrame({"feature": feature_names, "coefficient": coefs})
    .assign(abs_coef=lambda d: d["coefficient"].abs())
    .sort_values("abs_coef", ascending=False)
    .head(TOP_N_FEATURES)
    .sort_values("coefficient")
)

fig, ax = plt.subplots(figsize=(8, 6))
colors = ["#d73027" if c < 0 else "#1a9850" for c in coef_df["coefficient"]]
ax.barh(coef_df["feature"], coef_df["coefficient"], color=colors)
ax.axvline(0, color="black", linewidth=0.8)
ax.set_xlabel("Coefficient (log-odds)")
ax.set_title(f"Top {TOP_N_FEATURES} Logistic Regression Coefficients")
plt.tight_layout()
plt.show()

print("\nTop 10 features by absolute coefficient:")
print(coef_df.tail(10)[["feature", "coefficient"]].to_string(index=False))

## 8. Calibration

A well-calibrated model's predicted probability of home win should match the actual
observed win rate in each probability bin. The diagonal line represents perfect calibration.

In [ ]:
prob_true, prob_pred = calibration_curve(
    y_test, y_prob_test, n_bins=CALIBRATION_BINS, strategy="uniform"
)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

ax = axes[0]
ax.plot(prob_pred, prob_true, "o-", label="Logistic Regression")
ax.plot([0, 1], [0, 1], "k--", label="Perfect calibration")
ax.set_xlabel("Mean predicted probability")
ax.set_ylabel("Fraction of positives (actual home win rate)")
ax.set_title("Calibration Plot -- Test Set")
ax.legend()

ax2 = axes[1]
ax2.hist(y_prob_test[y_test == 0], bins=30, alpha=0.6, label="Away Win", density=True)
ax2.hist(y_prob_test[y_test == 1], bins=30, alpha=0.6, label="Home Win", density=True)
ax2.set_xlabel("Predicted probability of home win")
ax2.set_ylabel("Density")
ax2.set_title("Predicted Probability Distribution by Outcome")
ax2.legend()

plt.tight_layout()
plt.show()

print(f"Brier Score (test): {brier_score_loss(y_test, y_prob_test):.4f}  (lower is better; ~0.25 = random for balanced classes)")

## 9. Summary & Next Steps

| Metric | Train | Test |
|--------|-------|------|
| Accuracy | -- | -- |
| ROC-AUC | -- | -- |
| Brier Score | -- | -- |

*(Fill in after running)*

**Potential next steps:**
- Tune `C` with `LogisticRegressionCV` on the training set
- Try `penalty='l1'` with `solver='liblinear'` for automatic feature selection
- Add aggregate lineup features (e.g., mean lineup OPS, mean K%)
- Re-run `scraper.py` with a different time window (`trailing_N`, `full_prior_season`) and compare
- Compare against tree-based models (RandomForest, XGBoost)